In [ ]:
%%file finops_ml_consumer.py
from kafka import KafkaConsumer, KafkaProducer
import json, requests

BROKER = "broker:9092"
# UWAGA: W Twoim kodzie zmieniłeś port na 8002. Jeśli API działa na 8001, zmień to z powrotem.
API_URL = "http://localhost:8002/score" 

consumer = KafkaConsumer(
    'gcp_billing_events',
    bootstrap_servers=BROKER,
    group_id='ml-scoring-group',          # Grupa pozwala kontynuować od miejsca, w którym skończył
    auto_offset_reset='earliest',         # Czytaj dane od początku, jeśli brak offsetu
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

alert_producer = KafkaProducer(
    bootstrap_servers=BROKER,
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

print("✅ Konsument uruchomiony i nasłuchuje na 'gcp_billing_events'...")

try:
    for message in consumer:
        tx = message.value
        
        # --- MAPOWANIE ---
        features = {
            "billing_record_id": tx.get('billing_record_id', 'unknown'),
            "project_id": tx.get('project_id', 'unknown'),
            "service": tx.get('service', 'unknown'),
            "sku": tx.get('sku', 'unknown'),
            "usage_unit": tx.get('usage_unit', 'unknown'),
            "usage_amount": float(tx.get('usage_amount', 0)),
            "cost_usd": float(tx.get('cost_usd', 0)),
            "is_weekend": int(tx.get('is_weekend', 0)),
            "is_anomaly": int(tx.get('is_anomaly', 0))
        }
        
        # --- ODPYTANIE API ---
        try:
            response = requests.post(API_URL, json=features, timeout=2)
            result = response.json()
        except requests.RequestException:
            continue

        # --- TYLKO ALERTOWANIE (FILTROWANIE) ---
        if result.get('is_fraud'):
            # Wyciągamy dane prosto z oryginalnego obiektu tx
            service = tx.get('service', 'Unknown')
            cost = tx.get('cost_usd', 0.0)
            prob = result.get('fraud_probability', 0.0)
            tx_id = tx.get('billing_record_id', 'N/A')

            # Czytelny format alertu
            print(f"🚨 [FRAUD DETECTED] ID: {tx_id} | Serwis: {service} | Koszt: {cost} USD | Anomaly Score: {prob:.4f}")
            
            # Wysłanie alertu do Kafki
            alert = {
                **tx,
                'fraud_probability': prob,
                'alert_source': 'ml_model'
            }
            alert_producer.send('alerts', value=alert)

except KeyboardInterrupt:
    print("Zamykanie...")
finally:
    alert_producer.flush()
    alert_producer.close()
    consumer.close()